# 📊 Notebook 01 — Ranking Functions

**Functions covered:** `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()`, `NTILE()`, `PERCENT_RANK()`, `CUME_DIST()`

## Quick Reference

| Function | Ties | Gaps | Use when... |
|----------|------|------|-------------|
| `ROW_NUMBER()` | unique int per row | — | You need a unique sequential ID |
| `RANK()` | same rank | yes | Official leaderboard (Olympic style) |
| `DENSE_RANK()` | same rank | no | No skipped positions |
| `NTILE(n)` | splits evenly | — | Bucket/quartile assignment |
| `PERCENT_RANK()` | 0.0–1.0 | — | Relative standing as a percentage |
| `CUME_DIST()` | 0.0–1.0 | — | Cumulative distribution |


In [1]:
import duckdb, pandas as pd

employees = pd.read_csv("../data/employees.csv")
sales     = pd.read_csv("../data/sales.csv")
orders    = pd.read_csv("../data/orders.csv")
products  = pd.read_csv("../data/products.csv")
con = duckdb.connect()
con.register("employees", employees)
con.register("sales",     sales)

def sql(query):
    return con.execute(query).fetchdf()


---
## Exercise 1 · The Three Ranking Functions

**Question:** List every employee's `full_name`, `department`, `salary`, and their salary rank within the company using **all three** ranking functions (`ROW_NUMBER`, `RANK`, `DENSE_RANK`), ordered by rank ascending.

*Tip: Which employees share a rank? How do the results differ?*

In [2]:
# Your answer here
sql("""
SELECT
    full_name,
    department,
    salary,
    ROW_NUMBER() OVER (ORDER BY salary DESC) AS ROW_NUMBER_func,
    RANK() OVER(ORDER BY salary DESC) AS RANK_func,
    DENSE_RANK() OVER(ORDER BY salary DESC) AS DENSE_RANK_func
FROM employees
""")

,full_name,department,salary,ROW_NUMBER_func,RANK_func,DENSE_RANK_func
0,Margaret Brown,Engineering,192000,1,1,1
1,Joseph Jones,Engineering,173000,2,2,2
2,David Johnson,Engineering,172000,3,3,3
3,Christopher Miller,Engineering,172000,4,3,3
4,Anthony Davis,Finance,159000,5,5,4
5,David Jackson,Engineering,155000,6,6,5
6,Linda Miller,Engineering,155000,7,6,5
7,Lisa Martin,Engineering,149000,8,8,6
8,Mark Williams,Engineering,149000,9,8,6
9,Michael Lopez,Engineering,149000,10,8,6


<details><summary>💡 Solution</summary>

```sql
SELECT
    full_name,
    department,
    salary,
    ROW_NUMBER() OVER (ORDER BY salary DESC) AS row_num,
    RANK()       OVER (ORDER BY salary DESC) AS rnk,
    DENSE_RANK() OVER (ORDER BY salary DESC) AS dense_rnk
FROM employees
ORDER BY rnk;
```
</details>

---
## Exercise 2 · Top 3 Salaries Per Department

**Question:** Find the **top 3 highest-paid employees in each department**. Include ties. Show `full_name`, `department`, `salary`, and `dept_rank`.

*Tip: Use `DENSE_RANK()` so ties don't exclude employees.*

In [3]:
# Your answer here
sql("""
WITH data as (SELECT 
    full_name,
    department,
    salary,
    RANK() OVER(PARTITION BY department ORDER BY salary DESC) AS dept_rank
FROM employees

ORDER BY department DESC, salary DESC)

SELECT * FROM data WHERE dept_rank <= 3
    
""")

,full_name,department,salary,dept_rank
0,Patricia Miller,Sales,125000,1
1,Susan Moore,Sales,85000,2
2,Betty Davis,Sales,79000,3
3,John Anderson,Sales,79000,3
4,Nancy Williams,Sales,79000,3
5,Charles Gonzalez,Operations,135000,1
6,Joseph Rodriguez,Operations,122000,2
7,Ashley Anderson,Operations,104000,3
8,Mary Johnson,Marketing,124000,1
9,Sandra Garcia,Marketing,123000,2


<details><summary>💡 Solution</summary>

```sql
SELECT *
FROM (
    SELECT
        full_name,
        department,
        salary,
        DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dept_rank
    FROM employees
)
WHERE dept_rank <= 3
ORDER BY department, dept_rank;
```
</details>

---
## Exercise 3 · Second Highest Salary Per Department

**Question:** Find the employee(s) with the **2nd highest unique salary** in each department. Show `full_name`, `department`, `salary`.

*Tip: When using `DENSE_RANK`, the 2nd highest unique salary has rank = 2.*

In [4]:
# Your answer here
sql("""
WITH table_data AS (
    
    SELECT 
        full_name,
        department, 
        salary,
        DENSE_RANK() OVER(PARTITION BY department ORDER BY salary DESC) dense_rank_order,
    FROM 
        employees

    )

SELECT 
    full_name,
    department, 
    salary,
    dense_rank_order,
FROM 
    table_data
WHERE 
    dense_rank_order = 2
ORDER BY 
    department desc,
    salary desc
""")

,full_name,department,salary,dense_rank_order
0,Susan Moore,Sales,85000,2
1,Joseph Rodriguez,Operations,122000,2
2,Sandra Garcia,Marketing,123000,2
3,Christopher Martinez,Marketing,123000,2
4,James Brown,Marketing,123000,2
5,Sarah Gonzalez,HR,87000,2
6,Anthony Anderson,Finance,133000,2
7,Joseph Jones,Engineering,173000,2


<details><summary>💡 Solution</summary>

```sql
SELECT full_name, department, salary
FROM (
    SELECT
        full_name,
        department,
        salary,
        DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dept_rank
    FROM employees
)
WHERE dept_rank = 2
ORDER BY department;
```
</details>

---
## Exercise 4 · Salary Quartiles with NTILE

**Question:** Divide all employees into **4 salary quartiles** (1 = lowest, 4 = highest) using `NTILE(4)`. Show `full_name`, `salary`, and `salary_quartile`, sorted by quartile and salary.

*Bonus: How many employees fall into each quartile?*

In [5]:
# Your answer here
sql("""
SELECT 
    full_name,
    salary, 
    NTILE(4) OVER(ORDER BY salary DESC) AS salary_quartyle
FROM employees
ORDER BY salary_quartyle, salary
    
""")

,full_name,salary,salary_quartyle
0,Anthony Anderson,133000,1
1,Charles Gonzalez,135000,1
2,Betty Martin,147000,1
3,Lisa Martin,149000,1
4,Mark Williams,149000,1
5,Michael Lopez,149000,1
6,David Jackson,155000,1
7,Linda Miller,155000,1
8,Anthony Davis,159000,1
9,David Johnson,172000,1


<details><summary>💡 Solution</summary>

```sql
-- Main query
SELECT
    full_name,
    salary,
    NTILE(4) OVER (ORDER BY salary) AS salary_quartile
FROM employees
ORDER BY salary_quartile, salary;

-- Bonus: distribution count
-- SELECT salary_quartile, COUNT(*) AS cnt
-- FROM (
--     SELECT NTILE(4) OVER (ORDER BY salary) AS salary_quartile FROM employees
-- )
-- GROUP BY 1 ORDER BY 1;
```
</details>

---
## Exercise 5 · Top 10% Earners (PERCENT_RANK)

**Question:** Identify all employees who are in the **top 10% of salary** company-wide. Show `full_name`, `department`, `salary`, and `pct_rank` (as a percentage rounded to 1 decimal).

In [6]:
# Your answer here
sql("""
WITH data_table as (SELECT 
    full_name,
    department,
    salary,
    ROUND((PERCENT_RANK() OVER(ORDER BY salary DESC))*100, 1) AS pct_rank
FROM
    employees)

SELECT * FROM data_table
WHERE pct_rank <= 10
""")

,full_name,department,salary,pct_rank
0,Margaret Brown,Engineering,192000,0.0
1,Joseph Jones,Engineering,173000,2.0
2,David Johnson,Engineering,172000,4.1
3,Christopher Miller,Engineering,172000,4.1
4,Anthony Davis,Finance,159000,8.2


<details><summary>💡 Solution</summary>

```sql
SELECT full_name, department, salary, pct_rank
FROM (
    SELECT
        full_name,
        department,
        salary,
        ROUND(PERCENT_RANK() OVER (ORDER BY salary) * 100, 1) AS pct_rank
    FROM employees
)
WHERE pct_rank >= 90
ORDER BY salary DESC;
```
</details>

---
## Exercise 6 · Rank Sales Reps by Quarterly Revenue

**Question:** Rank each sales rep by their **total revenue in Q4 2023** within their region. Show `region`, `rep_name`, `total_q4_revenue`, and `region_rank`. Only include Q4 2023 data.

In [7]:
# Your answer here
sql("""
WITH ranking_region AS (
    SELECT 
        region, 
        rep_name, 
        SUM(amount) AS total_q4_revenue
    FROM 
        sales
    WHERE year = 2023 and quarter = 'Q1'
    GROUP BY region, rep_name
)
SELECT 
    region,
    rep_name, 
    total_q4_revenue,
    RANK() OVER(PARTITION BY region ORDER BY total_q4_revenue DESC) as region_rank
FROM ranking_region
ORDER BY region, region_rank
""")

,region,rep_name,total_q4_revenue,region_rank
0,Midwest,Christopher Anderson,126968.12,1
1,Midwest,John Anderson,79941.75,2
2,Northeast,Mary Taylor,197457.25,1
3,Northeast,Ashley Davis,97036.20,2
4,Northeast,Linda Lopez,79446.30,3
5,Southeast,Susan Moore,111213.93,1
6,Southeast,Christopher Wilson,40158.37,2
7,Southwest,Jennifer Brown,242608.58,1
8,Southwest,Patricia Miller,228313.32,2
9,Southwest,Nancy Williams,139151.35,3


<details><summary>💡 Solution</summary>

```sql
SELECT
    region,
    rep_name,
    total_q4_revenue,
    RANK() OVER (PARTITION BY region ORDER BY total_q4_revenue DESC) AS region_rank
FROM (
    SELECT
        region,
        rep_name,
        SUM(amount) AS total_q4_revenue
    FROM sales
    WHERE quarter = 'Q4' AND year = 2023
    GROUP BY region, rep_name
)
ORDER BY region, region_rank;
```
</details>

---
## Exercise 7 · Detect Duplicate Salaries

**Question:** Find all departments where **at least two employees have the exact same salary**. For each such salary, list the department, salary value, and all employee names sharing it.

*Tip: Use `COUNT()` as a window function to count peers with the same salary.*

In [8]:
# Your answer here
sql("""
WITH counts_table AS (
    SELECT 
        *,
        COUNT() OVER(PARTITION BY salary ORDER BY salary DESC) AS counts
    FROM employees
    )
SELECT 
    full_name,
    department,
    salary,
    counts
FROM counts_table
WHERE counts >= 2 
ORDER BY department, salary

""")

,full_name,department,salary,counts
0,Michael Moore,Engineering,85000,4
1,Charles Wilson,Engineering,92000,4
2,Daniel Moore,Engineering,92000,4
3,David Rodriguez,Engineering,92000,4
4,Michael Lopez,Engineering,149000,3
5,Lisa Martin,Engineering,149000,3
6,Mark Williams,Engineering,149000,3
7,David Jackson,Engineering,155000,2
8,Linda Miller,Engineering,155000,2
9,Christopher Miller,Engineering,172000,2


<details><summary>💡 Solution</summary>

```sql
SELECT department, salary, full_name
FROM (
    SELECT
        department,
        salary,
        full_name,
        COUNT(*) OVER (PARTITION BY department, salary) AS salary_count
    FROM employees
)
WHERE salary_count > 1
ORDER BY department, salary;
```
</details>

---
## Exercise 8 · Number Each Rep's Sales Chronologically

**Question:** For each sales rep, assign a sequential order number (`sale_number`) to their sales, starting from 1 for their earliest sale. Show `rep_name`, `sale_date`, `amount`, and `sale_number`.

In [9]:
# Your answer here
sql("""
SELECT 
    rep_name, 
    sale_date,
    amount,
    ROW_NUMBER() OVER(PARTITION BY rep_name ORDER BY sale_date DESC) AS sale_number
FROM
    sales
ORDER BY rep_name, sale_number
    """)

,rep_name,sale_date,amount,sale_number
0,Ashley Davis,2023-12-08,72160.23,1
1,Ashley Davis,2023-10-06,64712.92,2
2,Ashley Davis,2023-09-28,64712.92,3
3,Ashley Davis,2023-09-13,71125.63,4
4,Ashley Davis,2023-08-09,26557.41,5
...,...,...,...,...
265,Susan Moore,2022-10-17,71510.50,15
266,Susan Moore,2022-08-28,36094.33,16
267,Susan Moore,2022-05-29,78515.06,17
268,Susan Moore,2022-05-20,24928.53,18


<details><summary>💡 Solution</summary>

```sql
SELECT
    rep_name,
    sale_date,
    amount,
    ROW_NUMBER() OVER (PARTITION BY rep_name ORDER BY sale_date) AS sale_number
FROM sales
ORDER BY rep_name, sale_number;
```
</details>

---
## Ejercicio Extra 1 · Top de Salarios por Departamento

**Pregunta (Data Engineer):** Usando la tabla `employees`, encuentra a los 3 empleados mejor pagados por departamento usando `DENSE_RANK()`. Filtra fuera de la función de ventana.

<details><summary>💡 Solución Esperada</summary>

```sql
WITH Ranked AS (
  SELECT full_name, department, salary, DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) as rnk
  FROM employees
)
SELECT * FROM Ranked WHERE rnk <= 3;
```
</details>

In [10]:
# Escribe tu consulta aquí:
query = """
WITH dense_data AS (
    SELECT 
        full_name,
        department,
        salary,
        DENSE_RANK() OVER(PARTITION BY department ORDER BY salary DESC) AS rank
FROM employees
)

SELECT * 
FROM dense_data
WHERE rank <= 3
ORDER BY department, rank
"""
con.execute(query).df()

,full_name,department,salary,rank
0,Margaret Brown,Engineering,192000,1
1,Joseph Jones,Engineering,173000,2
2,David Johnson,Engineering,172000,3
3,Christopher Miller,Engineering,172000,3
4,Anthony Davis,Finance,159000,1
5,Anthony Anderson,Finance,133000,2
6,Susan Brown,Finance,119000,3
7,Matthew Wilson,Finance,119000,3
8,Sandra Thomas,HR,91000,1
9,Sarah Gonzalez,HR,87000,2


---
## Ejercicio Extra 2 · Paginación de Órdenes

**Pregunta (Data Engineer):** Un dashboard necesita mostrar las facturas cronológicamente pero paginadas de 50 en 50. Crea una columna con el número de factura absoluto usando `ROW_NUMBER()` sobre toda la tabla `orders`.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT order_id, order_date, total_amount, ROW_NUMBER() OVER (ORDER BY order_date, order_id) as absolute_order_num
FROM orders;
```
</details>

In [11]:
# Escribe tu consulta aquí:
sql("""
WITH TABLA_FACTURA AS (
    SELECT 
        *,
        ROW_NUMBER() OVER(ORDER BY order_date, order_id) AS rows
    FROM orders
)


SELECT * FROM TABLA_FACTURA
ORDER BY order_date
""")

,order_id,customer_id,customer_name,product_id,product_name,category,order_date,year,quarter,month,month_label,quantity,unit_price,total_amount,region,rows
0,8,13,Oceanic Airlines,8,Webcam HD,Electronics,2022-01-03,2022,Q1,1,2022-01,20,89.99,1799.80,Midwest,1
1,298,15,Cyberdyne Systems,1,Laptop Pro 15,Electronics,2022-01-10,2022,Q1,1,2022-01,2,1299.99,2599.98,Midwest,2
2,12,10,Vandelay Industries,11,Desk Lamp LED,Furniture,2022-01-17,2022,Q1,1,2022-01,11,39.99,439.89,Southeast,3
3,146,10,Vandelay Industries,5,Monitor 27in,Electronics,2022-01-17,2022,Q1,1,2022-01,18,349.99,6299.82,West,4
4,154,6,Pied Piper,18,Antivirus Suite,Software,2022-01-17,2022,Q1,1,2022-01,9,49.99,449.91,Midwest,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,266,1,Acme Corp,19,Video Conf License,Software,2023-12-19,2023,Q4,12,2023-12,19,199.99,3799.81,Northeast,296
296,190,13,Oceanic Airlines,12,Notebook Pack (10),Office Supplies,2023-12-22,2023,Q4,12,2023-12,9,14.99,134.91,Southwest,297
297,277,5,Hooli,1,Laptop Pro 15,Electronics,2023-12-23,2023,Q4,12,2023-12,10,1299.99,12999.90,Southeast,298
298,270,14,Waystar Royco,6,Mechanical Keyboard,Electronics,2023-12-24,2023,Q4,12,2023-12,2,129.99,259.98,Southeast,299


---
## Ejercicio Extra 3 · Percentiles de Costo de Producto

**Pregunta (Data Engineer):** Distribuye los productos en la tabla `products` en 10 deciles (usando `NTILE(10)`) basándote en su `cost`. ¿En qué decil caen los productos de la categoría 'Electronics'?

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT product_name, category, cost, NTILE(10) OVER (ORDER BY cost DESC) as cost_decile
FROM products
ORDER BY cost DESC;
```
</details>

In [12]:
# Escribe tu consulta aquí:
query = """
with data_table as (
SELECT *, NTILE(10) OVER(ORDER BY cost) AS Percentile
FROM products)

SELECT * 
FROM data_table
WHERE category = 'Electronics'
"""
con.execute(query).df()

,product_id,product_name,category,unit_price,cost,margin_pct,Percentile
0,2,Wireless Mouse,Electronics,29.99,12.0,60.0,3
1,15,USB-C Hub 7-port,Electronics,59.99,22.0,63.3,5
2,8,Webcam HD,Electronics,89.99,35.0,61.1,7
3,6,Mechanical Keyboard,Electronics,129.99,55.0,57.7,7
4,10,Printer All-in-One,Electronics,199.99,90.0,55.0,8
5,7,Noise Cancelling Headphones,Electronics,249.99,110.0,56.0,8
6,5,Monitor 27in,Electronics,349.99,175.0,50.0,9
7,1,Laptop Pro 15,Electronics,1299.99,780.0,40.0,10


---
## Ejercicio Extra 4 · Medianas y Cuartiles de Ventas

**Pregunta (Data Engineer):** Calcula el rank relativo a ventas usando `PERCENT_RANK()` en `sales` particionando por `category`. Identifica qué ventas superan el 90% (percentil 90).

<details><summary>💡 Solución Esperada</summary>

```sql
WITH PctRank AS (
  SELECT sale_id, category, amount, PERCENT_RANK() OVER (PARTITION BY category ORDER BY amount) as p_rank
  FROM sales
)
SELECT * FROM PctRank WHERE p_rank >= 0.9;
```
</details>

In [13]:
# Escribe tu consulta aquí:
query = """
WITH BASE_TABLE AS (
    SELECT *, 
    ROUND((PERCENT_RANK() OVER(PARTITION BY category ORDER BY amount)*100), 1) AS percet_rank
    FROM SALES
)

SELECT * FROM BASE_TABLE
WHERE percet_rank > 90
ORDER BY category, amount 
"""
con.execute(query).df()

,sale_id,rep_id,rep_name,region,category,amount,sale_date,year,quarter,month,month_label,percet_rank
0,252,27,Jennifer Brown,Southwest,Hardware,77819.42,2023-01-10,2023,Q1,1,2023-01,91.7
1,228,26,Ashley Davis,Northeast,Hardware,78437.12,2022-08-29,2022,Q3,8,2022-08,93.3
2,65,19,Betty Davis,West,Hardware,78838.40,2022-06-08,2022,Q2,6,2022-06,95.0
3,63,18,Linda Lopez,Northeast,Hardware,79446.30,2023-01-02,2023,Q1,1,2023-01,96.7
4,43,18,Linda Lopez,Northeast,Hardware,79580.62,2023-08-02,2023,Q3,8,2023-08,98.3
5,10,16,Christopher Anderson,Midwest,Hardware,80588.44,2022-05-12,2022,Q2,5,2022-05,100.0
6,73,19,Betty Davis,West,Services,73741.29,2023-02-13,2023,Q1,2,2023-02,90.9
7,261,27,Jennifer Brown,Southwest,Services,77479.17,2023-10-13,2023,Q4,10,2023-10,93.2
8,206,25,Susan Moore,Southeast,Services,79410.20,2023-05-26,2023,Q2,5,2023-05,95.5
9,253,27,Jennifer Brown,Southwest,Services,80367.59,2022-09-21,2022,Q3,9,2022-09,97.7


---
## Ejercicio Extra 5 · Empate en Ventas Reportadas

**Pregunta (Data Engineer):** Existen representantes de ventas con montos idénticos. Usa `RANK()` y `DENSE_RANK()` en la misma consulta sobre `sales` particionado por `region` y ordenado por `amount` para comparar cómo tratan los empates.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT rep_name, region, amount,
       RANK() OVER (PARTITION BY region ORDER BY amount DESC) as rnk,
       DENSE_RANK() OVER (PARTITION BY region ORDER BY amount DESC) as dense_rnk
FROM sales
ORDER BY region, amount DESC;
```
</details>

In [14]:
# Escribe tu consulta aquí:
query = """
WITH BASE_TABLE AS (
    SELECT 
        *, 
        RANK() OVER(PARTITION BY region ORDER BY amount) as ranking,
        DENSE_RANK() OVER(PARTITION BY region ORDER BY amount) as dense_ranking
    FROM sales
)

SELECT 
    *
FROM 
    BASE_TABLE
"""
con.execute(query).df()

,sale_id,rep_id,rep_name,region,category,amount,sale_date,year,quarter,month,month_label,ranking,dense_ranking
0,160,22,Christopher Wilson,Southeast,Hardware,10343.67,2023-09-04,2023,Q3,9,2023-09,1,1
1,161,22,Christopher Wilson,Southeast,Hardware,10343.67,2022-10-31,2022,Q4,10,2022-10,1,1
2,135,22,Christopher Wilson,Southeast,Support,12047.57,2023-12-02,2023,Q4,12,2023-12,3,2
3,220,25,Susan Moore,Southeast,Support,12478.44,2023-05-23,2023,Q2,5,2023-05,4,3
4,213,25,Susan Moore,Southeast,Support,14333.42,2023-11-07,2023,Q4,11,2023-11,5,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
265,102,20,John Anderson,Midwest,Training,76037.68,2023-09-20,2023,Q3,9,2023-09,41,32
266,90,20,John Anderson,Midwest,Hardware,76798.01,2023-06-08,2023,Q2,6,2023-06,42,33
267,91,20,John Anderson,Midwest,Support,76798.01,2023-11-06,2023,Q4,11,2023-11,42,33
268,10,16,Christopher Anderson,Midwest,Hardware,80588.44,2022-05-12,2022,Q2,5,2022-05,44,34


---
## Ejercicio Extra 6 · Análisis de ROW_NUMBER() en employees

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `ROW_NUMBER()` para la tabla `employees` particionado por `region` y ordenado por el métrico `amount` ASC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT region, amount, 
       ROW_NUMBER() OVER (PARTITION BY region ORDER BY amount ASC) as val
FROM employees
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 7 · Análisis de RANK() en products

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `RANK()` para la tabla `products` particionado por `category` y ordenado por el métrico `total_amount` DESC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT category, total_amount, 
       RANK() OVER (PARTITION BY category ORDER BY total_amount DESC) as val
FROM products
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 8 · Análisis de DENSE_RANK() en sales

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `DENSE_RANK()` para la tabla `sales` particionado por `department` y ordenado por el métrico `salary` ASC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT department, salary, 
       DENSE_RANK() OVER (PARTITION BY department ORDER BY salary ASC) as val
FROM sales
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 9 · Análisis de NTILE(5) en orders

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `NTILE(5)` para la tabla `orders` particionado por `city` y ordenado por el métrico `margin_pct` DESC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT city, margin_pct, 
       NTILE(5) OVER (PARTITION BY city ORDER BY margin_pct DESC) as val
FROM orders
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 10 · Análisis de CUME_DIST() en employees

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `CUME_DIST()` para la tabla `employees` particionado por `year` y ordenado por el métrico `unit_price` ASC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT year, unit_price, 
       CUME_DIST() OVER (PARTITION BY year ORDER BY unit_price ASC) as val
FROM employees
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 11 · Análisis de PERCENT_RANK() en products

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `PERCENT_RANK()` para la tabla `products` particionado por `month_label` y ordenado por el métrico `cost` DESC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT month_label, cost, 
       PERCENT_RANK() OVER (PARTITION BY month_label ORDER BY cost DESC) as val
FROM products
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 12 · Análisis de ROW_NUMBER() en sales

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `ROW_NUMBER()` para la tabla `sales` particionado por `region` y ordenado por el métrico `amount` ASC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT region, amount, 
       ROW_NUMBER() OVER (PARTITION BY region ORDER BY amount ASC) as val
FROM sales
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 13 · Análisis de RANK() en orders

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `RANK()` para la tabla `orders` particionado por `category` y ordenado por el métrico `total_amount` DESC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT category, total_amount, 
       RANK() OVER (PARTITION BY category ORDER BY total_amount DESC) as val
FROM orders
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 14 · Análisis de DENSE_RANK() en employees

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `DENSE_RANK()` para la tabla `employees` particionado por `department` y ordenado por el métrico `salary` ASC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT department, salary, 
       DENSE_RANK() OVER (PARTITION BY department ORDER BY salary ASC) as val
FROM employees
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 15 · Análisis de NTILE(5) en products

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `NTILE(5)` para la tabla `products` particionado por `city` y ordenado por el métrico `margin_pct` DESC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT city, margin_pct, 
       NTILE(5) OVER (PARTITION BY city ORDER BY margin_pct DESC) as val
FROM products
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 16 · Análisis de CUME_DIST() en sales

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `CUME_DIST()` para la tabla `sales` particionado por `year` y ordenado por el métrico `unit_price` ASC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT year, unit_price, 
       CUME_DIST() OVER (PARTITION BY year ORDER BY unit_price ASC) as val
FROM sales
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 17 · Análisis de PERCENT_RANK() en orders

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `PERCENT_RANK()` para la tabla `orders` particionado por `month_label` y ordenado por el métrico `cost` DESC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT month_label, cost, 
       PERCENT_RANK() OVER (PARTITION BY month_label ORDER BY cost DESC) as val
FROM orders
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 18 · Análisis de ROW_NUMBER() en employees

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `ROW_NUMBER()` para la tabla `employees` particionado por `region` y ordenado por el métrico `amount` ASC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT region, amount, 
       ROW_NUMBER() OVER (PARTITION BY region ORDER BY amount ASC) as val
FROM employees
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 19 · Análisis de RANK() en products

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `RANK()` para la tabla `products` particionado por `category` y ordenado por el métrico `total_amount` DESC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT category, total_amount, 
       RANK() OVER (PARTITION BY category ORDER BY total_amount DESC) as val
FROM products
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 20 · Análisis de DENSE_RANK() en sales

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `DENSE_RANK()` para la tabla `sales` particionado por `department` y ordenado por el métrico `salary` ASC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT department, salary, 
       DENSE_RANK() OVER (PARTITION BY department ORDER BY salary ASC) as val
FROM sales
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 21 · Análisis de NTILE(5) en orders

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `NTILE(5)` para la tabla `orders` particionado por `city` y ordenado por el métrico `margin_pct` DESC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT city, margin_pct, 
       NTILE(5) OVER (PARTITION BY city ORDER BY margin_pct DESC) as val
FROM orders
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 22 · Análisis de CUME_DIST() en employees

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `CUME_DIST()` para la tabla `employees` particionado por `year` y ordenado por el métrico `unit_price` ASC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT year, unit_price, 
       CUME_DIST() OVER (PARTITION BY year ORDER BY unit_price ASC) as val
FROM employees
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 23 · Análisis de PERCENT_RANK() en products

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `PERCENT_RANK()` para la tabla `products` particionado por `month_label` y ordenado por el métrico `cost` DESC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT month_label, cost, 
       PERCENT_RANK() OVER (PARTITION BY month_label ORDER BY cost DESC) as val
FROM products
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 24 · Análisis de ROW_NUMBER() en sales

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `ROW_NUMBER()` para la tabla `sales` particionado por `region` y ordenado por el métrico `amount` ASC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT region, amount, 
       ROW_NUMBER() OVER (PARTITION BY region ORDER BY amount ASC) as val
FROM sales
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 25 · Análisis de RANK() en orders

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `RANK()` para la tabla `orders` particionado por `category` y ordenado por el métrico `total_amount` DESC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT category, total_amount, 
       RANK() OVER (PARTITION BY category ORDER BY total_amount DESC) as val
FROM orders
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 26 · Análisis de DENSE_RANK() en employees

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `DENSE_RANK()` para la tabla `employees` particionado por `department` y ordenado por el métrico `salary` ASC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT department, salary, 
       DENSE_RANK() OVER (PARTITION BY department ORDER BY salary ASC) as val
FROM employees
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 27 · Análisis de NTILE(5) en products

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `NTILE(5)` para la tabla `products` particionado por `city` y ordenado por el métrico `margin_pct` DESC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT city, margin_pct, 
       NTILE(5) OVER (PARTITION BY city ORDER BY margin_pct DESC) as val
FROM products
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 28 · Análisis de CUME_DIST() en sales

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `CUME_DIST()` para la tabla `sales` particionado por `year` y ordenado por el métrico `unit_price` ASC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT year, unit_price, 
       CUME_DIST() OVER (PARTITION BY year ORDER BY unit_price ASC) as val
FROM sales
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 29 · Análisis de PERCENT_RANK() en orders

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `PERCENT_RANK()` para la tabla `orders` particionado por `month_label` y ordenado por el métrico `cost` DESC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT month_label, cost, 
       PERCENT_RANK() OVER (PARTITION BY month_label ORDER BY cost DESC) as val
FROM orders
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 30 · Análisis de ROW_NUMBER() en employees

**Pregunta (Data Engineer):** Como Data Engineer, revisa la calidad de datos generando un ranking con `ROW_NUMBER()` para la tabla `employees` particionado por `region` y ordenado por el métrico `amount` ASC.

<details><summary>💡 Solución Esperada</summary>

```sql
SELECT region, amount, 
       ROW_NUMBER() OVER (PARTITION BY region ORDER BY amount ASC) as val
FROM employees
LIMIT 100;
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## 🎯 Summary

- `ROW_NUMBER()` → always unique; use for pagination or deduplication.
- `RANK()` → ties get same rank, next rank skips (1, 2, 2, 4).
- `DENSE_RANK()` → ties get same rank, no skipping (1, 2, 2, 3) — best for "top-N per group".
- `NTILE(n)` → distributes rows into n buckets; great for quartile/decile analysis.
- `PERCENT_RANK()` / `CUME_DIST()` → relative positioning (0.0–1.0).

➡️ **Next:** `02_offset_functions.ipynb` — LAG, LEAD, FIRST_VALUE, LAST_VALUE